# Step 1: Data Understanding
**Personal portfolio project: Insurance Customer Retention Analysis**

*This is a personal portfolio project. The dataset is anonymised and treated as fictional: carriers appear only as numeric codes, and the figures are used for demonstration, not as statements about any real company.*

## Business context
A Canadian personal-lines insurance brokerage places auto and property policies with 16 insurance carriers.
Every cancelled policy is lost commission and lost lifetime value, so management wants to know **which policies are most likely to cancel and where the retention team should focus**.

## Project objectives
1. Measure how often policy terms are cancelled, at renewal and mid-term.
2. Identify customer, product and regional segments with above-average cancellation.
3. Turn the findings into practical retention recommendations.

## Analytical questions
| # | Question |
|---|---|
| Q1 | What share of policy terms cancel, and is it mostly at renewal or mid-term? |
| Q2 | Which lines of business cancel most? |
| Q3 | Do clients holding both auto and property policies (bundled) stay longer? |
| Q4 | How does cancellation change with customer tenure? |
| Q5 | Are higher-premium policies more likely to cancel? |
| Q6 | Which provinces and carriers have the highest cancellation? |
| Q7 | For auto policies, does driver age or vehicle count matter? |

This notebook covers the first step: receiving the data and understanding its structure before any changes are made.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RAW, PROC, IMG = Path('../data/raw'), Path('../data/processed'), Path('../images')
IMG.mkdir(exist_ok=True); PROC.mkdir(parents=True, exist_ok=True)
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.spines.top': False, 'axes.spines.right': False})
COLOR = '#2E6F95'

def save(fname):
    plt.tight_layout(); plt.savefig(IMG / fname, dpi=120); plt.show()

def load_raw():
    # EPIC_ID is built from client names: drop it on load so it never appears in any output
    terms = pd.read_excel(RAW / '1-insurance_2025_policy_terms.xlsx', sheet_name='result').drop(columns='EPIC_ID')
    demo = pd.read_excel(RAW / '2-insurance_2025_demographics.xlsx')
    return terms, demo

## 1.1 Files received

In [2]:
for f in sorted(RAW.glob('*.xlsx')):
    print(f'{f.name:40s} {f.stat().st_size / 1e6:6.1f} MB')
for f in ['1-insurance_2025_policy_terms.xlsx', '2-insurance_2025_demographics.xlsx']:
    print(f, '-> sheets:', pd.ExcelFile(RAW / f).sheet_names)

0-insurance_2025_dictionary.xlsx            0.0 MB
1-insurance_2025_policy_terms.xlsx         14.8 MB
2-insurance_2025_demographics.xlsx          4.7 MB


1-insurance_2025_policy_terms.xlsx -> sheets: ['result', 'Demographic']
2-insurance_2025_demographics.xlsx -> sheets: ['result']


Three files were received: a data dictionary, a policy terms table and a demographics table.
The policy terms workbook also holds a copy of the demographics sheet, so only its `result` sheet is used.

In [3]:
terms, demo = load_raw()
print('Policy terms:', terms.shape)
print('Demographics:', demo.shape)

Policy terms: (100000, 14)
Demographics: (100000, 10)


## 1.2 Data dictionary
### Policy terms (one row per policy term)

| Field | Description |
|---|---|
| Client_id | Unique client identifier |
| Policy_id | Policy identifier; terms of the same policy share it |
| Policy_Term_id | Unique policy term identifier (primary key) |
| Carrier_id | Insurance carrier that underwrites the policy |
| EPIC_ID | Internal client code (removed for privacy) |
| Client_First_Written | Date the client's first policy was written; used for tenure |
| Term_Effective_Date | Start date of the term |
| Term_Expiration_Date | End date of the term (renewal or cancellation) |
| Term_Status | NEW, REN (renewal), REI (reissued) = active; CAN, LAP (lapsed), INA (inactive) = cancelled |
| Line_of_Business | Product type, e.g. AUTO, HOME, CNDO (condo), TPAC (tenant package) |
| Term_Premium | Premium for the term (CAD) |
| FSA | First three characters of the postal code |
| Province | Province |
| CAN_REN | 1 if the policy cancelled at renewal |
| CAN_MT | 1 if the policy cancelled mid-term |

### Demographics (auto policies)
| Field | Description |
|---|---|
| Policy_Term_id | Join key to policy terms |
| max_birthdate_max | Birth date of the youngest driver |
| min_birthdate_min | Birth date of the oldest driver |
| max_max_of_drivernumber | Number of drivers |
| max_max_of_vehiclenumber | Number of vehicles |
| max_max_of_yearslicensed | Maximum years licensed of any driver |
| max_tpbi | Liability coverage limit |
| max_ap | Comprehensive deductible (if purchased) |
| max_col | Collision deductible |
| min_applicantbirthdate | Birth date of the applicant |

Lines of business are grouped into **Auto** (AUTO, MOTO) and **Property** (all others).

## 1.3 Structure and data types

In [4]:
terms.dtypes.to_frame('dtype').T

,Client_id,Policy_id,Policy_Term_id,Carrier_id,Client_First_Written,Term_Effective_Date,Term_Expiration_Date,Term_Status,Line_of_Business,Term_Premium,FSA,Province,CAN_REN,CAN_MT
dtype,int64,int64,int64,int64,datetime64[us],datetime64[us],datetime64[us],str,str,float64,str,str,int64,int64


In [5]:
terms.drop(columns=['Client_id', 'FSA']).head()

,Policy_id,Policy_Term_id,Carrier_id,Client_First_Written,Term_Effective_Date,Term_Expiration_Date,Term_Status,Line_of_Business,Term_Premium,Province,CAN_REN,CAN_MT
0,2340359,2340359,65584,2023-06-07,2023-06-07,2024-06-07,NEW,AUTO,"2,923.00",ON,0,0
1,966284,2350359,65575,2017-07-08,2023-07-08,2023-07-08,CAN,AUTO,"2,008.00",ON,1,0
2,841281,2357366,65567,2016-07-20,2023-07-20,2024-07-20,REN,AUTO,"1,804.00",ON,0,0
3,649488,2388173,65567,2015-07-29,2023-08-15,2023-11-15,CAN,AUTO,"3,214.00",ON,0,1
4,303359,2463114,65577,2015-10-15,2023-10-13,2024-10-13,REN,AUTO,"1,970.87",ON,0,0


In [6]:
demo.dtypes.to_frame('dtype').T

,Policy_Term_id,max_birthdate_max,min_birthdate_min,max_max_of_drivernumber,max_max_of_vehiclenumber,max_max_of_yearslicensed,max_tpbi,max_ap,max_col,min_applicantbirthdate
dtype,int64,datetime64[us],datetime64[us],float64,float64,float64,float64,float64,float64,datetime64[us]


## 1.4 Granularity and keys

In [7]:
checks = pd.Series({
    'Policy terms: unique Policy_Term_id': terms['Policy_Term_id'].is_unique,
    'Demographics: unique Policy_Term_id': demo['Policy_Term_id'].is_unique,
    'Distinct clients': terms['Client_id'].nunique(),
    'Distinct policies': terms['Policy_id'].nunique(),
    'Distinct carriers': terms['Carrier_id'].nunique(),
    'Terms matched in demographics': terms['Policy_Term_id'].isin(demo['Policy_Term_id']).sum(),
    'Auto terms matched in demographics': terms.loc[terms['Line_of_Business'].str.strip().isin(['AUTO', 'MOTO']), 'Policy_Term_id'].isin(demo['Policy_Term_id']).sum(),
})
checks.to_frame('value')

,value
Policy terms: unique Policy_Term_id,True
Demographics: unique Policy_Term_id,True
Distinct clients,76417
Distinct policies,91342
Distinct carriers,16
Terms matched in demographics,24400
Auto terms matched in demographics,15354


In [8]:
terms[['Client_First_Written', 'Term_Effective_Date', 'Term_Expiration_Date']].agg(['min', 'max']).T

,min,max
Client_First_Written,1976-01-28,2025-08-27
Term_Effective_Date,2022-11-13,2025-10-04
Term_Expiration_Date,2022-11-13,2026-10-04


## 1.5 First observations
- Each row in the policy terms table is **one policy term**; `Policy_Term_id` is a valid primary key.
- 100,000 terms belong to about 76,000 clients, so many clients hold more than one policy or term.
- Terms start between late 2022 and October 2025.
- Only about a quarter of terms have a matching demographics record, so driver analysis will cover a subset of auto policies.
- The target variable is built from `CAN_REN` and `CAN_MT`: a term is **churned** if either flag is 1.

Next step: explore distributions and data quality (Notebook 02).